In [5]:
import ROOT as r
from itertools import combinations
from neutrinosolver import *

# opens file and tree
f = r.TFile("actual_data/ttbar_5k.root")
tree = f.Get("Events")

nEvents = tree.GetEntries()

# classes for muon, electron, jets

class MyMuon(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, iso=0.0, charge=0):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.isolation = iso
        self.charge = charge
    
    
    def IsIsolated(self, relcut=0.1):
        if self.Pt() == 0:
            return False
        return self.isolation < relcut

class MyElectron(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, iso=0.0, charge=0, cutBased=0):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.isolation = iso
        self.charge = charge
        self.cutBased = cutBased

    def IsIsolated(self, relcut=0.1):
        if self.Pt() == 0:
            return False
        return self.isolation < relcut

class MyJet(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, btag=0.0, jetid=False):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.btag = btag
        self.jetid = jetid
        self.is_btagged = False

    def IsBTagged(self, threshold):
        return self.btag > threshold

    def HasJetID(self):
        return (self.jetid & 2) != 0
    
    

# histogram

h_Mwh_vs_Mth = r.TH2F(
    "h_Mwh_vs_Mth",
    "M(W_h) vs M(t_h); M(t_h) [GeV]; M(W_h) [GeV]",
    50, 0, 500,
    50, 0, 300
)


# weighted error
h_Mwh_vs_Mth.Sumw2()


# analysis cuts

cuts = {
    "Muon": {
        "pt_min": 30.0,
        "eta_max": 2.4,
        "iso_max": 0.15
    },
    "Electron": {
        "pt_min": 30.0,
        "eta_max": 2.4,
        "cutBased": 4,
        "iso_max": 0.15
        
    },
    "Jets": {
        "pt_min": 30.0,
        "eta_max": 2.4,
        "btag": 0.3040
    },
    "MET": {
        "pt_min": 20.0
    }
}

cuts["Trigger"] = {
    "muon": ["HLT_IsoMu27"],
    "electron": ["HLT_Ele27_WPTight_Gsf", "HLT_Ele32_WPTight_Gsf"]
}

cutflow = {
    "total": 0,

    # triggers
    "pass_mu_trigger": 0,
    "pass_ele_trigger": 0,

    # lepton selection
    "exactly_1_lepton": 0,

    # MET
    "pass_MET": 0,

    # jets
    "3jets": 0,
    "4plus_jets": 0,

    # b-tag categories
    "4pj_2b": 0,
    "4pj_1b": 0
}

#finds the real bs that come from a top
    
def has_top_ancestor(idx):
    while idx >= 0:
        pdgId = tree.GenPart_pdgId[idx]
        if abs(pdgId) == 6:
            return True
        idx = tree.GenPart_genPartIdxMother[idx]
    return False

# event loop
for event in range(nEvents):

    cutflow["total"] += 1

    tree.GetEntry(event)
    weight = tree.Generator_weight


    # ---------------------------------------------------------
    # Generator-level hadronic top decay products
    # ---------------------------------------------------------

    gen_b_had = None
    gen_w_quarks = []

    for i in range(tree.nGenPart):

        pdgId = abs(tree.GenPart_pdgId[i])

        # only quarks
        if pdgId not in [1, 2, 3, 4, 5]:
            continue

        mother_idx = tree.GenPart_genPartIdxMother[i]

        if mother_idx < 0:
            continue

        mother_pdg = abs(tree.GenPart_pdgId[mother_idx])

        # -------------------------------------------------
        # b quark from a top
        # -------------------------------------------------

        if pdgId == 5:

            if not has_top_ancestor(mother_idx):
                continue

            gen_b_had = r.TLorentzVector()
            gen_b_had.SetPtEtaPhiM(
                tree.GenPart_pt[i],
                tree.GenPart_eta[i],
                tree.GenPart_phi[i],
                tree.GenPart_mass[i]
            )

        # light quarks from a W coming from a top

        elif pdgId in [1, 2, 3, 4]:

            # immediate mother must be a W
            if mother_pdg != 24:
                continue

            # W must ultimately come from a top
            if not has_top_ancestor(mother_idx):
                continue

            qvec = r.TLorentzVector()
            qvec.SetPtEtaPhiM(
                tree.GenPart_pt[i],
                tree.GenPart_eta[i],
                tree.GenPart_phi[i],
                tree.GenPart_mass[i]
            )

            gen_w_quarks.append(qvec)

    # Require truth objects

    if len(gen_w_quarks) != 2:
        continue

    if gen_b_had is None:
        continue
        
    #MET cuts
    if not all([
        tree.Flag_goodVertices,
        tree.Flag_globalSuperTightHalo2016Filter,
        tree.Flag_HBHENoiseFilter,
        tree.Flag_HBHENoiseIsoFilter,
        tree.Flag_EcalDeadCellTriggerPrimitiveFilter,
        tree.Flag_BadPFMuonFilter,
        tree.Flag_BadPFMuonDzFilter,
        tree.Flag_eeBadScFilter,
        tree.Flag_ecalBadCalibFilter
        ]):
        continue


    # trigger cuts
    passes_mu_trigger = any(
        getattr(tree, trig, False) for trig in cuts["Trigger"]["muon"]
        )

    passes_ele_trigger = any(
        getattr(tree, trig, False) for trig in cuts["Trigger"]["electron"]
    )   

    if passes_mu_trigger:
        cutflow["pass_mu_trigger"] += 1

    if passes_ele_trigger:
        cutflow["pass_ele_trigger"] += 1

    # muons

    muons = []
    for mu_idx in range(tree.nMuon):
        mu = MyMuon(
            tree.Muon_pt[mu_idx],
            tree.Muon_eta[mu_idx],
            tree.Muon_phi[mu_idx],
            tree.Muon_mass[mu_idx],   
            tree.Muon_miniPFRelIso_all[mu_idx],
            tree.Muon_charge[mu_idx] #unsure
            )
        muons.append(mu)

    #iso_muons = [m for m in muons if m.IsIsolated(MuonRelIsoCut)]

    iso_muons = [
    m for m in muons
    if m.Pt() > cuts["Muon"]["pt_min"]
    and abs(m.Eta()) < cuts["Muon"]["eta_max"]
    and m.isolation < cuts["Muon"]["iso_max"]
    ]



    iso_muons = sorted(iso_muons, key=lambda m: m.Pt(), reverse=True)

    #if len(iso_muons) >= 2 and iso_muons[0].Pt() > MuonPtCut:
        #dimu = iso_muons[0] + iso_muons[1]
        #h_Mmumu.Fill(dimu.M(), weight)


    # electrons
    electrons = []
    for ele_idx in range(tree.nElectron):
        ele = MyElectron(
            tree.Electron_pt[ele_idx],
            tree.Electron_eta[ele_idx],
            tree.Electron_phi[ele_idx],
            tree.Electron_mass[ele_idx],   
            tree.Electron_miniPFRelIso_all[ele_idx],  
            tree.Electron_charge[ele_idx],
            tree.Electron_cutBased[ele_idx]
            )
        electrons.append(ele)

    
    iso_electrons = [
        e for e in electrons
        if e.Pt() > cuts["Electron"]["pt_min"]
        and abs(e.Eta()) < cuts["Electron"]["eta_max"]
        and e.cutBased >= cuts["Electron"]["cutBased"]
        and e.isolation < cuts["Electron"]["iso_max"] 
        ]

    #iso_electrons = [e for e in electrons if e.IsIsolated(ElectronRelIsoCut)]
    iso_electrons = sorted(iso_electrons, key=lambda e: e.Pt(), reverse=True)

    #if len(iso_electrons) >= 2 and iso_electrons[0].Pt() > ElectronPtCut:
    #   diele = iso_electrons[0] + iso_electrons[1]
    #  h_Mee.Fill(diele.M(), weight)


    # MET from tree
    MET  = tree.MET_pt
    phi  = tree.MET_phi


    METx = MET * r.TMath.Cos(phi)
    METy = MET * r.TMath.Sin(phi)

    n_iso_mu  = len(iso_muons)
    n_iso_ele = len(iso_electrons)

    if n_iso_mu == 1 and n_iso_ele == 0:
        if not passes_mu_trigger:
            continue
        selected_lepton = iso_muons[0]

    elif n_iso_ele == 1 and n_iso_mu == 0:
        if not passes_ele_trigger:
            continue
        selected_lepton = iso_electrons[0]

    else:
        continue
    cutflow["exactly_1_lepton"] += 1
    passes_MET = MET > cuts["MET"]["pt_min"]

    if not passes_MET:
        continue
    cutflow["pass_MET"] += 1

    
    # jets

    jets = []
    for jet_idx in range(tree.nJet):
        jet = MyJet(
            tree.Jet_pt[jet_idx],
            tree.Jet_eta[jet_idx],
            tree.Jet_phi[jet_idx],
            tree.Jet_mass[jet_idx],    
            tree.Jet_btagDeepFlavB[jet_idx],    
            tree.Jet_jetId[jet_idx] 
            )
        jets.append(jet)

    
    def deltaR(obj1, obj2):
        return obj1.DeltaR(obj2)

    good_jets = [
    j for j in jets
    if j.Pt() > cuts["Jets"]["pt_min"]
    and abs(j.Eta()) < cuts["Jets"]["eta_max"]
    and j.HasJetID()
    and deltaR(j, selected_lepton) > 0.4
]

    #good_jets = [j for j in jets if j.HasJetID() and j.Pt() > JetPtCut]
    good_jets = sorted(good_jets, key=lambda j: j.Pt(), reverse=True)
    
    

    # flagging b-tagged jets
    for j in good_jets:
        # j.is_btagged = j.IsBTagged(BTagThreshold)
        j.is_btagged = j.IsBTagged(cuts["Jets"]["btag"])

    #bjets = [j for j in good_jets if j.IsBTagged(BTagThreshold)] - old usage before fagging btagged jets
    bjets = [j for j in good_jets if j.is_btagged]
    bjets = sorted(bjets, key=lambda j: j.Pt(), reverse=True)

    n_jets = len(good_jets)
    n_bjets = len(bjets)

    #part of nusolver
    # choose a b jet temporarily for neutrino solving
    # usually use leading b jet
    if len(bjets) < 1:
        continue

    b_for_nu = bjets[0]

    # MET covariance matrix
    sigma2 = np.array([
        [100, 0],
        [0, 100]
    ])

    try:
        nusol = singleNeutrinoSolution(
            b_for_nu,
            selected_lepton,
            METx,
            METy,
            sigma2
        )

        neutrinos = []

        nu_vec = nusol.nu

        nu = r.TLorentzVector()
        nu.SetPxPyPzE(
            nu_vec[0],
            nu_vec[1],
            nu_vec[2],
            math.sqrt(nu_vec[0]**2 + nu_vec[1]**2 + nu_vec[2]**2)
        )

        neutrinos.append(nu)


    except Exception as e:
        continue


    
    
    
    #4-6 jets

    if 4 <= n_jets <= 6:
        cutflow["4plus_jets"] += 1

        if n_bjets >= 2:
            cutflow["4pj_2b"] += 1

            b1, b2 = bjets[0], bjets[1]
            light_jets = [j for j in good_jets if not j.is_btagged]

            if len(light_jets) < 2:
                continue

            for nu in neutrinos:
                best_pair = None
                best_diff = 1e9

                for j1_tmp, j2_tmp in combinations(light_jets, 2):
                    W_tmp = j1_tmp + j2_tmp
                    diff = abs(W_tmp.M() - 80.4)

                    if diff < best_diff:
                        best_diff = diff
                        best_pair = (j1_tmp, j2_tmp)

                if best_pair is None:
                    continue

                j1, j2 = best_pair
                W_had = j1 + j2

                for b_lep, b_had in [(b1,b2),(b2,b1)]:

                    # leptonic top
                    t_lep = selected_lepton + nu + b_lep
                    # hadronic top
                    t_had = W_had + b_had
                    h_Mwh_vs_Mth.Fill(t_had.M(), W_had.M(), weight)
                    



# histograms

print("\n=== CUTFLOW ===")
for key, val in cutflow.items():
    print(f"{key:15s}: {val}")



#probability 2d hist

c_Mwh_vs_Mth = r.TCanvas("c_Mwh_vs_Mth", "M(W_h) vs M(t_h)", 800, 600)

if h_Mwh_vs_Mth.Integral() > 0:
    h_Mwh_vs_Mth.Scale(1.0 / h_Mwh_vs_Mth.Integral("width"))

def top_probability(M_top, M_W, hist):
    """
    Returns the probability density at (M_top, M_W)
    from the normalized 2D histogram.
    """

    xbin = hist.GetXaxis().FindBin(M_top)
    ybin = hist.GetYaxis().FindBin(M_W)

    return hist.GetBinContent(xbin, ybin)


#p = top_probability(173.2, 81.1, h_Mwh_vs_Mth)
#print(p)

h_Mwh_vs_Mth.Draw("COLZ")   # important for 2D histograms

c_Mwh_vs_Mth.Update()


import os
os.makedirs("plots", exist_ok=True)
c_Mwh_vs_Mth.SaveAs("plots/h_Mwh_vs_Mth.png")



=== CUTFLOW ===
total          : 5000
pass_mu_trigger: 1016
pass_ele_trigger: 818
exactly_1_lepton: 1534
pass_MET       : 1397
3jets          : 0
4plus_jets     : 535
4pj_2b         : 321
4pj_1b         : 0


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_Mwh_vs_Mth
Info in <TCanvas::Print>: png file plots/h_Mwh_vs_Mth.png has been created
